In [1]:
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

from utils.funciones_minio import crear_cliente_minio, bajar_minio,bajar_mapa_minio,buscar_todos_los_archivos
from utils.config import PATH_PRIMARIOS_LIMPIO

In [4]:
import pandas as pd
import io
from PIL import Image
from IPython.display import display

cliente = crear_cliente_minio()


df_prueba = bajar_minio(cliente,"cleaned/dataset_vision/Cocina","batch_1.parquet")

for i in range(min(5, len(df_prueba))):
    fila = df_prueba.iloc[i]
    id_piso = fila['id']
    bytes_img = fila['imagen_bytes']
    
    img = Image.open(io.BytesIO(bytes_img))

    display(img)

ReadTimeoutError: HTTPSConnectionPool(host='minio.fdi.ucm.es', port=443): Read timed out.

In [ ]:
def auditar_dataset_minio():
    """
    Recorre los Parquets particionados en MinIO y cuenta el total de imágenes por clase.
    """
    cliente = crear_cliente_minio()
    
    clases = ['Cocina', 'Dormitorio', 'Salón', 'Banyo', 'Comedor']
    conteo_clase = {}
    total_general = 0
    for clase in clases:
        total_clase = 0
        objetos = buscar_todos_los_archivos(cliente,f"cleaned/dataset_vision/{clase}")
        for obj in objetos:
            df_temp = bajar_minio(cliente,f"cleaned/dataset_vision/{clase}",obj)
            
            cantidad_chunk = len(df_temp)
            total_clase += cantidad_chunk
        total_general += total_clase
        conteo_clase[clase] = total_clase

    print("\n" + "="*40)
    print("BALANCE FINAL DEL DATASET")
    print("="*40)
    for clase, total in conteo_clase.items():
        porcentaje = (total / total_general) * 100
        print(f"    {clase.capitalize().ljust(15)}: {total:,} imgs ({porcentaje:.1f}%)")
            
    print("-" * 40)
    print(f"    TOTAL GLOBAL   : {total_general:,} imgs")
    print("="*40)
        
    return conteo_clase

distribucion = auditar_dataset_minio()
print(distribucion)